In [ ]:
# 1. 导入模块、读取配置和路径
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

_base = Path('/home/intern_fjq_2026').resolve()
_current = Path.cwd().resolve()
_candidates = [
    _base / 'Projects' / 'chinese-wwm-roberta',
    _base.parent / 'intern_fjq_2026' / 'Projects' / 'chinese-wwm-roberta',
    _current,
    _current.parent,
]
ROOT = next((path for path in _candidates if (path / 'src').is_dir()), None)
assert ROOT is not None, 'Repository root was not found'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.activation_rank import (
    finalize_activation_rank_run,
    load_activation_rank_config,
    plot_rank_curves,
    plot_rank_heatmap,
    preflight_activation_rank,
    run_pilot_stage,
    run_rank_analysis_stage,
    run_rank_shards_stage,
    run_sample_stage,
    run_wo_mechanism_stage,
    validate_pilot_outputs,
    validate_rank_analysis_outputs,
    validate_wo_mechanism_outputs,
)

CONFIG_PATH = ROOT / 'configs' / 'activation_rank.yaml'
config = load_activation_rank_config(CONFIG_PATH)
RUN_DIR = Path(config['output']['run_directory']).expanduser().resolve()
display({'config': str(CONFIG_PATH), 'run_directory': str(RUN_DIR)})

In [ ]:
# 2. 服务器、GPU、磁盘、checkpoint与文本preflight
preflight = preflight_activation_rank(config)
display(preflight)
assert preflight['status'].eq('ok').all(), preflight

In [ ]:
# 3. 构建或严格复用sample manifest
sample_dir = run_sample_stage(config)
sample_manifest = pd.read_parquet(sample_dir / 'sample_manifest.parquet')
display(sample_manifest.head())
display(sample_manifest.groupby('shard')['token_count'].agg(['count', 'sum']))

In [ ]:
# 4. hooks、精度、batch size与吞吐pilot
pilot_dir = run_pilot_stage(config)
pilot_manifest = validate_pilot_outputs(pilot_dir)
display(pd.Series(pilot_manifest))
display(pd.read_parquet(pilot_dir / 'batch_benchmark.parquet'))
display(pd.read_parquet(pilot_dir / 'precision_metrics.parquet'))

In [ ]:
# 5. 运行缺失的rank shards；已验证分片自动跳过
# kernel重启或某一shard失败后，只需重跑cell 1与本cell。
try:
    moments_dir = run_rank_shards_stage(config)
except RuntimeError:
    audit_candidates = [RUN_DIR / 'moments' / 'norm_audit_5m.parquet', RUN_DIR / 'moments' / 'norm_audit.parquet']
    for audit_path in audit_candidates:
        if audit_path.is_file():
            display(pd.read_parquet(audit_path).sort_values(['blocking', 'position_passed', 'population', 'site'], ascending=[False, True, True, True]))
    raise
moments_manifest = json.loads((moments_dir / 'manifest.json').read_text(encoding='utf-8'))
display(pd.Series(moments_manifest))
display(pd.DataFrame(moments_manifest['norm_audit']['failed_blocking_positions']))
norm_audit = pd.read_parquet(moments_dir / 'norm_audit.parquet')
display(norm_audit.sort_values(['blocking', 'position_passed', 'population', 'site'], ascending=[False, True, True, True]))

In [ ]:
# 6. 合并moments、FP64特征分解与稳定性/坍缩分析
analysis_dir = run_rank_analysis_stage(config)
display(pd.Series(validate_rank_analysis_outputs(analysis_dir)))
rank_metrics = pd.read_parquet(analysis_dir / 'rank_metrics.parquet')
compression_metrics = pd.read_parquet(analysis_dir / 'compression_metrics.parquet')
display(rank_metrics)
display(compression_metrics)

In [ ]:
# 7. W^O协方差恒等式、方向分解与反事实机制分析
mechanism_dir = run_wo_mechanism_stage(config)
display(pd.Series(validate_wo_mechanism_outputs(mechanism_dir)))
display(pd.read_parquet(mechanism_dir / 'wo_summary.parquet'))
display(pd.read_parquet(mechanism_dir / 'wo_mechanism.parquet').head(24))

In [ ]:
# 8. 图表、结论表与最终run manifest
completed_dir = finalize_activation_rank_run(config)
final_manifest = json.loads((completed_dir / 'manifest.json').read_text(encoding='utf-8'))
display(pd.Series(final_manifest))
display(plot_rank_heatmap(completed_dir / 'analysis'))
display(plot_rank_curves(completed_dir / 'analysis'))
display(pd.read_parquet(completed_dir / 'compression_metrics.parquet'))